### The S3 Ingestionator V2
Ingests Cloudtrail logs directly from S3 into a dataframe. This version uses multiple threads to process days in parallel with circuit breakers for memory utilization. Part of the DUNE project (https://github.com/opendr-io/dune) for hunting threats that are resistant to conventional detection. Run this in the same region!  Populate the target S3 URI below. I'm assuming credentials are already populated for the AWS client because this notebook is probably running in the same region as the data.

In [ ]:
import gc, io, gzip, json, os
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
import boto3
from botocore.exceptions import ClientError
import pandas as pd
import psutil


In [ ]:
!aws s3 ls

In [ ]:
S3_URI = "s3://aws-cloudtrail-logs-"

FILTER_EVENT_SOURCE = None    # e.g. "sts.amazonaws.com"; set both to exclude an event type
FILTER_EVENT_NAME = None      # e.g. "AssumeRole"
MAX_MEMORY_MB = 12048
MAX_PARALLEL_DAYS = 1


In [ ]:
# Interrogate the URI and calculate the data volume there
def _parse_s3_uri(s3_uri: str):
    parsed = urlparse(s3_uri.rstrip("/"))
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    prefix = parsed.path.lstrip("/")
    return parsed.netloc, (prefix + "/") if prefix else ""

S3_BUCKET, S3_PREFIX = _parse_s3_uri(S3_URI)

def iter_s3_objects(s3_client, bucket, prefix):
    continuation_token = None
    while True:
        kwargs = {"Bucket": bucket, "Prefix": prefix}
        if continuation_token:
            kwargs["ContinuationToken"] = continuation_token
        page = s3_client.list_objects_v2(**kwargs)
        yield from page.get("Contents", [])
        if not page.get("IsTruncated"):
            break
        continuation_token = page.get("NextContinuationToken")

s3 = boto3.client("s3")

all_objs = []
for obj in iter_s3_objects(s3, S3_BUCKET, S3_PREFIX):
    if obj["Key"].endswith(".json.gz"):
        all_objs.append(obj)

print(f"Found {len(all_objs)} files under {S3_URI}")

total_bytes = sum(obj["Size"] for obj in all_objs)
total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print(f"Total size: {total_bytes:,} bytes")
print(f"About {total_mb:.2f} MB  ({total_gb:.3f} GB)")


In [ ]:
# Enumerate fields that are found by the json ingestor
def list_cloudtrail_fields(s3_uri, limit=10):
    bucket, prefix = _parse_s3_uri(s3_uri)
    s3 = boto3.client("s3")
    field_types = {}   # field -> set of python type names seen
    field_subkeys = {} # field -> set of sub-keys (dict fields only)
    files_read = 0
    for obj in iter_s3_objects(s3, bucket, prefix):
        if not obj["Key"].endswith(".json.gz"):
            continue
        body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
        with gzip.GzipFile(fileobj=io.BytesIO(body)) as gz:
            payload = json.loads(gz.read().decode("utf-8"))
            for rec in payload.get("Records", []):
                for field, val in rec.items():
                    field_types.setdefault(field, set()).add(type(val).__name__)
                    if isinstance(val, dict):
                        field_subkeys.setdefault(field, set()).update(val.keys())
        files_read += 1
        if limit and files_read >= limit:
            break
    scalars, dot_children, blobs = [], {}, []
    for field in sorted(field_types):
        non_null = field_types[field] - {"NoneType"}
        if "dict" in non_null:
            dot_children[field] = sorted(field_subkeys.get(field, []))
        elif "list" in non_null:
            blobs.append(field)
        else:
            scalars.append(field)
    return scalars, dot_children, blobs

scalars, dot_children, blobs = list_cloudtrail_fields(S3_URI, limit=5)

print(f"Scalar fields ({len(scalars)}):")
for f in scalars:
    print(f"  - {f}")

print(f"Dot-navigable fields ({len(dot_children)}):")
for f, subkeys in dot_children.items():
    print(f"  - {f}: {chr(44).join(subkeys)}")

print(f"Blob fields / lists ({len(blobs)}):")
for f in blobs:
    print(f"  - {f}")

all_fields = (
    scalars
    + [f"{parent}.{child}" for parent, subkeys in dot_children.items() for child in subkeys]
    + blobs
)
print(f"All selectable fields ({len(all_fields)}):")
for f in all_fields:
    print(f"  {f}")

print("")
print("CLOUDTRAIL_SELECT_COLS = [")
for f in all_fields:
    print(f'    "{f}",')
print("]")


In [ ]:
CLOUDTRAIL_SELECT_COLS = [
    "awsRegion",
    "eventCategory",
    "eventID",
    "eventName",
    "eventSource",
    "eventTime",
    "eventType",
    "eventVersion",
    "managementEvent",
    "readOnly",
    "recipientAccountId",
    "requestID",
    "sharedEventID",
    "sourceIPAddress",
    "userAgent",
    "vpcEndpointAccountId",
    "vpcEndpointId",
    "userIdentity.accessKeyId",
    "userIdentity.accountId",
    "userIdentity.arn",
    "userIdentity.invokedBy",
    "userIdentity.principalId",
    "userIdentity.sessionContext",
    "userIdentity.type",
    "resources"
]

In [ ]:
# Define eventSource/eventName pairs to drop during ingest.
# FILTER_FILE may point to a CSV/TSV with eventSource,eventName columns, or no header.
FILTER_FILE = None  # e.g. "cloudtrail_filters.csv"
FILTER_EVENT_PAIRS = {
     ("sts.amazonaws.com", "AssumeRole"),
    # ("ec2.amazonaws.com", "DescribeInstances"),
}

def _normalize_filter_pair(event_source, event_name):
    event_source = (event_source or "").strip()
    event_name = (event_name or "").strip()
    if not event_source or not event_name:
        return None
    return event_source, event_name

def _load_filter_file(path):
    import csv
    if not path:
        return set()

    with open(path, "r", encoding="utf-8-sig", newline="") as handle:
        rows = [line for line in handle if line.strip() and not line.lstrip().startswith("#")]

    if not rows:
        return set()

    sample = "".join(rows[:5])
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",	")
    except csv.Error:
        dialect = csv.excel

    reader = csv.reader(rows, dialect)
    first_row = next(reader, None)
    if first_row is None:
        return set()

    first_normalized = [cell.strip().lower().replace(" ", "") for cell in first_row]
    has_header = "eventsource" in first_normalized and "eventname" in first_normalized
    filter_pairs = set()

    if has_header:
        source_idx = first_normalized.index("eventsource")
        name_idx = first_normalized.index("eventname")
    else:
        source_idx = 0
        name_idx = 1
        pair = _normalize_filter_pair(
            first_row[source_idx] if len(first_row) > source_idx else None,
            first_row[name_idx] if len(first_row) > name_idx else None,
        )
        if pair:
            filter_pairs.add(pair)

    for row in reader:
        pair = _normalize_filter_pair(
            row[source_idx] if len(row) > source_idx else None,
            row[name_idx] if len(row) > name_idx else None,
        )
        if pair:
            filter_pairs.add(pair)

    return filter_pairs

def _record_matches_filter(record, filter_pairs):
    if not filter_pairs:
        return False
    return (record.get("eventSource"), record.get("eventName")) in filter_pairs

CLOUDTRAIL_FILTER_PAIRS = set(FILTER_EVENT_PAIRS) | _load_filter_file(FILTER_FILE)
if FILTER_EVENT_SOURCE and FILTER_EVENT_NAME:
    pair = _normalize_filter_pair(FILTER_EVENT_SOURCE, FILTER_EVENT_NAME)
    if pair:
        CLOUDTRAIL_FILTER_PAIRS.add(pair)

print(f"Dropping records matching {len(CLOUDTRAIL_FILTER_PAIRS):,} eventSource/eventName filter pair(s)")


In [ ]:
proc = psutil.Process(os.getpid())

def rss_mb():
    return proc.memory_info().rss / 1024 ** 2

def _date_from_s3_key(key: str):
    parts = key.split("/")
    for i in range(len(parts) - 2):
        year, month, day = parts[i:i + 3]
        if len(year) == 4 and year.isdigit() and len(month) == 2 and month.isdigit() and len(day) == 2 and day.isdigit():
            return f"{year}-{month}-{day}"
    return "unknown"

def _get_field(record: dict, field: str):
    value = record
    for part in field.split("."):
        if not isinstance(value, dict) or part not in value:
            return None
        value = value[part]
    return value

def _extract_fields(record: dict, select_cols):
    if not select_cols:
        return record
    result = {}
    for field in select_cols:
        val = _get_field(record, field)
        if isinstance(val, (dict, list)):
            val = json.dumps(val, default=str)
        result[field] = val
    return result

def _objects_by_day(objects):
    grouped = {}
    for obj in objects:
        grouped.setdefault(_date_from_s3_key(obj["Key"]), []).append(obj)
    return grouped

def load_cloudtrail_daily_frames_from_s3_uri(
    s3_uri: str,
    region_name=None,
    suffix=".json.gz",
    limit=None,
    select_cols=None,
    datetime_col="eventTime",
    filter_out=None,
    max_memory_mb=None,
    objects=None,
    existing_daily_frames=None,
    max_workers=1,
) -> dict:
    bucket, prefix = _parse_s3_uri(s3_uri)
    s3 = boto3.client("s3", region_name=region_name)

    daily_frames = dict(existing_daily_frames or {})
    stopped_early = False
    stop_reason = None
    _stop_event = threading.Event()

    def _load_cloudtrail_day(log_day, day_objects):
        worker_s3 = boto3.client("s3", region_name=region_name)
        chunks = []
        for obj in day_objects:
            if _stop_event.is_set():
                break

            key = obj["Key"]

            if max_memory_mb and rss_mb() > max_memory_mb:
                raise MemoryError(f"RSS {rss_mb():.1f} MB exceeded {max_memory_mb} MB while loading {log_day}")

            body = worker_s3.get_object(Bucket=bucket, Key=key)["Body"].read()
            with gzip.GzipFile(fileobj=io.BytesIO(body)) as gz:
                try:
                    payload = json.loads(gz.read().decode("utf-8"))
                except json.JSONDecodeError:
                    continue

            rows = []
            for record in payload.get("Records", []):
                if _record_matches_filter(record, filter_out):
                    continue
                rows.append(_extract_fields(record, select_cols) if select_cols else record)

            if not rows:
                continue

            chunk = pd.DataFrame.from_records(rows) if select_cols else pd.json_normalize(rows, sep=".")
            del body, payload, rows
            gc.collect()
            chunks.append(chunk)

        if not chunks:
            return log_day, pd.DataFrame(columns=select_cols or None)

        day_frame = pd.concat(chunks, ignore_index=True)
        if datetime_col in day_frame.columns:
            day_frame[datetime_col] = pd.to_datetime(day_frame[datetime_col], utc=True, errors="coerce")
        return log_day, day_frame

    if objects is None:
        objects = [obj for obj in iter_s3_objects(s3, bucket, prefix) if obj["Key"].endswith(suffix)]
    filtered_objects = [obj for obj in objects if obj["Key"].endswith(suffix)]
    if limit:
        filtered_objects = filtered_objects[:limit]
    objects_by_day = _objects_by_day(filtered_objects)
    days_to_load = [day for day in sorted(objects_by_day) if day not in daily_frames]

    for log_day in sorted(objects_by_day):
        if log_day in daily_frames:
            print(f"Skipping {log_day}: already loaded {len(daily_frames[log_day]):,} rows")

    worker_count = max(1, int(max_workers or 1))
    if days_to_load:
        print(f"Loading {len(days_to_load)} days with {worker_count} worker(s)")

    executor = ThreadPoolExecutor(max_workers=worker_count)
    try:
        futures = {
            executor.submit(_load_cloudtrail_day, day, objects_by_day[day]): day
            for day in days_to_load
        }

        for future in as_completed(futures):
            if future.cancelled():
                continue
            day = futures[future]
            try:
                loaded_day, day_frame = future.result()
            except ClientError as err:
                code = err.response.get("Error", {}).get("Code")
                if code in {"ExpiredToken", "InvalidToken", "TokenRefreshRequired"}:
                    stop_reason = f"AWS token expired while loading {day}"
                    print(f"Ingest stopped: {stop_reason}. Keeping completed daily DataFrames.")
                    stopped_early = True
                    _stop_event.set()
                    for pending in futures:
                        pending.cancel()
                    continue
                raise
            except MemoryError as err:
                stop_reason = str(err)
                print(f"Ingest stopped: {stop_reason}. Keeping completed daily DataFrames.")
                stopped_early = True
                _stop_event.set()
                for pending in futures:
                    pending.cancel()
                continue

            daily_frames[loaded_day] = day_frame
            print(f"  {loaded_day} complete: {len(day_frame):,} rows")
            del day_frame
            gc.collect()

    except KeyboardInterrupt:
        _stop_event.set()
        for pending in futures:
            pending.cancel()
        raise
    finally:
        executor.shutdown(wait=True, cancel_futures=True)

    globals()["cloudtrail_ingest_stopped_early"] = stopped_early
    globals()["cloudtrail_ingest_stop_reason"] = stop_reason

    if stopped_early:
        print("Ingest stopped before all days were loaded. Refresh credentials and rerun this cell to continue.")

    return daily_frames

def load_cloudtrail_from_s3_uri(*args, **kwargs) -> pd.DataFrame:
    daily_frames = load_cloudtrail_daily_frames_from_s3_uri(*args, **kwargs)
    if not daily_frames:
        select_cols = kwargs.get("select_cols")
        return pd.DataFrame(columns=select_cols or None)
    return pd.concat(daily_frames.values(), ignore_index=True)


In [ ]:
# Benchmark day-level ingest parallelism — same days for every worker count.

BENCHMARK_WORKER_COUNTS = [3, 4, 5, 6, 7]


def _sample_pressure(stop_event, samples, interval=1.0):
    psutil.cpu_percent(interval=None)
    while not stop_event.wait(interval):
        samples.append({
            "cpu_percent": psutil.cpu_percent(interval=None),
            "rss_mb": rss_mb(),
            "system_memory_percent": psutil.virtual_memory().percent,
        })


def _load_cloudtrail_day_for_benchmark(log_day, day_objects, download_log):
    worker_s3 = boto3.client("s3", region_name="us-east-1")
    chunks = []

    for obj in day_objects:
        if MAX_MEMORY_MB and rss_mb() > MAX_MEMORY_MB:
            raise MemoryError(f"RSS {rss_mb():.1f} MB exceeded {MAX_MEMORY_MB} MB while loading {log_day}")

        key = obj["Key"]
        t0 = time.perf_counter()
        body = worker_s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()
        download_log.append((len(body), time.perf_counter() - t0))

        with gzip.GzipFile(fileobj=io.BytesIO(body)) as gz:
            payload = json.loads(gz.read().decode("utf-8"))

        rows = []
        for record in payload.get("Records", []):
            if _record_matches_filter(record, CLOUDTRAIL_FILTER_PAIRS):
                continue
            rows.append(_extract_fields(record, CLOUDTRAIL_SELECT_COLS))

        if rows:
            chunks.append(pd.DataFrame.from_records(rows))

        del body, payload, rows
        gc.collect()

    if not chunks:
        return log_day, pd.DataFrame(columns=CLOUDTRAIL_SELECT_COLS)

    day_frame = pd.concat(chunks, ignore_index=True)
    if "eventTime" in day_frame.columns:
        day_frame["eventTime"] = pd.to_datetime(day_frame["eventTime"], utc=True, errors="coerce")
    return log_day, day_frame


objects_by_day = _objects_by_day([obj for obj in all_objs if obj["Key"].endswith(".json.gz")])
benchmark_days = sorted(objects_by_day)
benchmark_bytes = sum(obj["Size"] for day in benchmark_days for obj in objects_by_day[day])

print(f"Benchmark pool: {len(benchmark_days)} days, {benchmark_bytes / 1024:.1f} KB compressed")
print(f"Days: {', '.join(benchmark_days)}")

cloudtrail_benchmark_results = []

for worker_count in BENCHMARK_WORKER_COUNTS:
    print()
    print(f"Benchmark {worker_count} workers: loading {len(benchmark_days)} days")
    start_rss = rss_mb()
    stop_event = threading.Event()
    pressure_samples = []
    download_log = []  # (bytes, seconds) per S3 GET, appended by workers
    sampler = threading.Thread(target=_sample_pressure, args=(stop_event, pressure_samples), daemon=True)
    sampler.start()
    start = time.perf_counter()
    stopped_early = False
    stop_reason = None
    completed_days = 0
    completed_rows = 0

    try:
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = {
                executor.submit(_load_cloudtrail_day_for_benchmark, day, objects_by_day[day], download_log): day
                for day in benchmark_days
            }
            for future in as_completed(futures):
                day = futures[future]
                try:
                    loaded_day, day_frame = future.result()
                except ClientError as err:
                    code = err.response.get("Error", {}).get("Code")
                    if code in {"ExpiredToken", "InvalidToken", "TokenRefreshRequired"}:
                        stop_reason = f"AWS token expired while benchmarking {day}"
                        stopped_early = True
                        for pending in futures:
                            pending.cancel()
                        continue
                    raise
                except MemoryError as err:
                    stop_reason = str(err)
                    stopped_early = True
                    for pending in futures:
                        pending.cancel()
                    continue

                completed_days += 1
                completed_rows += len(day_frame)
                del day_frame
                gc.collect()
    finally:
        stop_event.set()
        sampler.join(timeout=2)

    elapsed = time.perf_counter() - start
    avg_cpu = sum(s["cpu_percent"] for s in pressure_samples) / len(pressure_samples) if pressure_samples else None
    max_cpu = max((s["cpu_percent"] for s in pressure_samples), default=None)
    max_rss = max((s["rss_mb"] for s in pressure_samples), default=rss_mb())
    max_system_mem = max((s["system_memory_percent"] for s in pressure_samples), default=psutil.virtual_memory().percent)
    mb_compressed = benchmark_bytes / 1024 ** 2

    total_dl_bytes = sum(b for b, _ in download_log)
    total_dl_worker_seconds = sum(s for _, s in download_log)
    s3_wall_mb_s = (total_dl_bytes / 1024 ** 2) / elapsed if elapsed > 0 else None
    s3_conn_mb_s = (total_dl_bytes / 1024 ** 2) / total_dl_worker_seconds if total_dl_worker_seconds > 0 else None
    dl_pct = total_dl_worker_seconds / (elapsed * worker_count) * 100 if elapsed > 0 else None

    result = {
        "workers": worker_count,
        "days": len(benchmark_days),
        "days_completed": completed_days,
        "rows": completed_rows,
        "elapsed_seconds": elapsed,
        "seconds_per_day": elapsed / completed_days if completed_days else None,
        "mb_compressed": mb_compressed,
        "s3_wall_mb_s": s3_wall_mb_s,
        "s3_conn_mb_s": s3_conn_mb_s,
        "dl_pct_of_worker_time": dl_pct,
        "rss_start_mb": start_rss,
        "rss_max_mb": max_rss,
        "rss_delta_mb": max_rss - start_rss,
        "cpu_avg_percent": avg_cpu,
        "cpu_max_percent": max_cpu,
        "system_memory_max_percent": max_system_mem,
        "stopped_early": stopped_early,
        "stop_reason": stop_reason,
    }
    cloudtrail_benchmark_results.append(result)

    print(
        f"Benchmark {worker_count} workers complete: "
        f"{completed_days}/{len(benchmark_days)} days, {completed_rows:,} rows, {elapsed:.1f}s | "
        f"S3 {s3_wall_mb_s:.3f} MB/s wall, {s3_conn_mb_s:.3f} MB/s per-conn, "
        f"dl {dl_pct:.0f}% of worker time | "
        f"avg CPU {avg_cpu if avg_cpu is not None else 0:.1f}%, max RSS {max_rss:.1f} MB"
    )
    if stop_reason:
        print(f"Stop reason: {stop_reason}")
        break

cloudtrail_benchmark_report = pd.DataFrame(cloudtrail_benchmark_results)
cloudtrail_benchmark_report


In [ ]:
FILTER_EVENT_SOURCE = None    # e.g. "sts.amazonaws.com"; set both to exclude an event type
FILTER_EVENT_NAME = None      # e.g. "AssumeRole"
MAX_MEMORY_MB = 12048
MAX_PARALLEL_DAYS = 3

In [ ]:
# Extract account ID from AWSLogs/{account_id}/CloudTrail/... prefix
_prefix_parts = [p for p in S3_PREFIX.split("/") if p]
_account_id = _prefix_parts[1] if len(_prefix_parts) > 1 and _prefix_parts[0] == "AWSLogs" else "unknown"

def _daily_var(account_id, date):
    return f"cloudtrail_{account_id}_{date.replace("-", "_")}"

# Collect already-loaded daily frames from named globals so re-runs skip completed days
_prefix = f"cloudtrail_{_account_id}_"
existing_daily_frames = {
    key[len(_prefix):].replace("_", "-"): val
    for key, val in globals().items()
    if key.startswith(_prefix) and isinstance(val, pd.DataFrame) and len(key) == len(_prefix) + 10
}

cloudtrail_by_day = load_cloudtrail_daily_frames_from_s3_uri(
    S3_URI,
    region_name="us-east-1",
    filter_out=CLOUDTRAIL_FILTER_PAIRS,
    max_memory_mb=MAX_MEMORY_MB,
    objects=all_objs,
    existing_daily_frames=existing_daily_frames,
    max_workers=MAX_PARALLEL_DAYS,
    select_cols=CLOUDTRAIL_SELECT_COLS,
)

# Save each day as a named global so token expiry mid-run does not lose completed work
for _day, _frame in cloudtrail_by_day.items():
    globals()[_daily_var(_account_id, _day)] = _frame

print(f"Loaded {len(cloudtrail_by_day)} daily DataFrames:")
for _day in sorted(cloudtrail_by_day):
    _var = _daily_var(_account_id, _day)
    print(f"  {_var}: {len(cloudtrail_by_day[_day]):,} rows")
if globals().get("cloudtrail_ingest_stopped_early"):
    print(f"Stopped early: {globals().get("cloudtrail_ingest_stop_reason")}")
    print("Refresh credentials and rerun this cell to resume.")


In [ ]:
# Merge daily DataFrames into cloudtrail
_prefix = f"cloudtrail_{_account_id}_"
_daily_frames = {
    key[len(_prefix):].replace("_", "-"): val
    for key, val in globals().items()
    if key.startswith(_prefix) and isinstance(val, pd.DataFrame) and len(key) == len(_prefix) + 10
}

if not _daily_frames:
    raise ValueError("No daily DataFrames found - run the ingest cell first.")

_expected_rows = sum(len(f) for f in _daily_frames.values())

cloudtrail = pd.concat(
    [_daily_frames[d] for d in sorted(_daily_frames)],
    ignore_index=True,
)

assert not cloudtrail.empty, "Merged DataFrame cloudtrail is empty"
assert len(cloudtrail) == _expected_rows, (
    f"Row count mismatch: merged {len(cloudtrail):,} rows but daily frames sum to {_expected_rows:,}"
)
print(f"cloudtrail: {len(_daily_frames)} days, {len(cloudtrail):,} rows, {cloudtrail.shape[1]} columns")
cloudtrail.head()


In [ ]:
# Free memory by deleting daily DataFrames now that cloudtrail is merged
_prefix = f"cloudtrail_{_account_id}_"
_daily_keys = [
    key for key in list(globals())
    if key.startswith(_prefix) and len(key) == len(_prefix) + 10
]
for _key in _daily_keys:
    del globals()[_key]
gc.collect()
print(f"Deleted {len(_daily_keys)} daily DataFrames: {chr(10).join(_daily_keys)}")


In [ ]:
# Describe the dataframe that has been produced for inspection.

if cloudtrail.empty:
    print("cloudtrail is empty.")
else:
    total_rows = len(cloudtrail)
    nan_counts = cloudtrail.isna().sum()

    missing_cols = [c for c in CLOUDTRAIL_SELECT_COLS if c not in cloudtrail.columns]
    if missing_cols:
        print(f"Columns in CLOUDTRAIL_SELECT_COLS not ingested ({len(missing_cols)}):")
        for c in missing_cols:
            print(f"  - {c}")
    else:
        print("All CLOUDTRAIL_SELECT_COLS present.")

    all_nan = nan_counts[nan_counts == total_rows].index.tolist()
    mostly_nan = nan_counts[(nan_counts / total_rows >= 0.95) & (nan_counts < total_rows)].index.tolist()

    print()
    print(f"Total rows: {total_rows:,}")

    if all_nan:
        print()
        print(f"Columns entirely NaN ({len(all_nan)}):")
        for c in all_nan:
            print(f"  - {c}")

    if mostly_nan:
        print()
        print(f"Columns >= 95% NaN ({len(mostly_nan)}):")
        for c in mostly_nan:
            print(f"  - {c}: {nan_counts[c] / total_rows * 100:.1f}% NaN")

    print()
    cloudtrail.info(verbose=True)
